# MediSense-Cardio -- Full Degradation Sweep v2 (adds PR / QRS / QT / QTc)

**What this notebook does:** re-runs the same study as v1 (`MediSense_Cardio_FullSweep_v1.ipynb`) --
the same **50**-record sample (same pool, same seed), the same **6 degradation axes x 5 levels + clean**
(31 variants/record, ~1,550 variants total) -- but this time **keeps the digitized signal** long enough to
measure **PR interval, QRS duration, QT interval and QTc**, not just heart rate. v1's results.csv is left
untouched; this writes to a separate `results_v2.csv`.

**Why re-run instead of reuse:** v1 discarded the digitized waveform right after computing heart rate, so
nothing from v1 can be reused -- this has to digitize every image again from scratch. Interval extraction
itself is cheap (CPU, NeuroKit2); the GPU digitization time (~20.3s/image, ~10.5 hours total) is unchanged.

**This notebook is resumable**, exactly like v1: state is checkpointed to Google Drive
(`MyDrive/MediSense-Cardio-Study/`), sub-batched so a disconnect loses at most a few minutes.

**How to use it:** `Runtime -> Run all`. It will ask you to authorize Google Drive access the first time.
When it stops (time budget reached, or all done), just **run it again later** -- it picks up automatically.
Repeat until the last cell prints `ALL VARIANTS COMPLETE`.


## Step 0 -- Mount Google Drive and install base tools


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
STUDY_DIR = '/content/drive/MyDrive/MediSense-Cardio-Study'
os.makedirs(STUDY_DIR, exist_ok=True)
os.makedirs(f'{STUDY_DIR}/images_clean', exist_ok=True)
print(f"Study checkpoint directory: {STUDY_DIR}")


In [ ]:
!pip install -q wfdb neurokit2 opencv-python-headless pandas numpy scipy matplotlib
print("Base tools installed.")


## Step 1 -- Draw (or reload) the fixed stratified sample

A larger 500-record **pool** is drawn once and saved to Drive (so scope can be widened again later without
re-drawing). The actual study sample used everywhere below is a fixed **50-record subset** of that pool,
chosen deterministically (same random seed every run) -- so results accumulate against one consistent sample.


In [ ]:
import pandas as pd, ast, ssl, urllib.request

POOL_PATH = f'{STUDY_DIR}/sample_500.csv'
SCOPE_N = 50  # trimmed from 500 to fit the 2026-09-28 deadline
rng_seed = 42

if os.path.exists(POOL_PATH):
    pool = pd.read_csv(POOL_PATH, dtype={'filename_lr': str})
    print(f"Loaded existing record pool from Drive: {len(pool)} records.")
else:
    # PhysioNet serves the open PTB-XL files directly over HTTPS -- fetching just the
    # two small label CSVs (not the ~1.8GB archive) is all that's needed to pick the pool.
    BASE = 'https://physionet.org/files/ptb-xl/1.0.3/'
    ctx = ssl.create_default_context()

    def fetch_csv(name):
        with urllib.request.urlopen(BASE + name, context=ctx) as r:
            with open(name, 'wb') as f:
                f.write(r.read())
        return pd.read_csv(name)

    database = fetch_csv('ptbxl_database.csv')
    scp_statements = fetch_csv('scp_statements.csv').rename(columns={'Unnamed: 0': 'code'})
    code_to_class = dict(zip(scp_statements['code'], scp_statements['diagnostic_class']))

    def dominant_class(scp_codes_str):
        codes = ast.literal_eval(scp_codes_str)
        if not codes:
            return None
        top_code = max(codes, key=codes.get)
        return code_to_class.get(top_code)

    database['diagnostic_class'] = database['scp_codes'].apply(dominant_class)
    database['is_abnormal'] = database['diagnostic_class'].notna() & (database['diagnostic_class'] != 'NORM')

    n_total = len(database)
    n_abnormal = int(database['is_abnormal'].sum())
    print(f"PTB-XL: {n_total} records total, {n_abnormal} ({100*n_abnormal/n_total:.1f}%) classified abnormal.")

    POOL_N = 500
    normal_pool_all = database[~database['is_abnormal']]
    abnormal_pool_all = database[database['is_abnormal']]
    pool = pd.concat([
        normal_pool_all.sample(n=POOL_N // 2, random_state=rng_seed),
        abnormal_pool_all.sample(n=POOL_N - POOL_N // 2, random_state=rng_seed),
    ]).reset_index(drop=True)[['ecg_id', 'filename_lr', 'diagnostic_class', 'is_abnormal']]

    pool.to_csv(POOL_PATH, index=False)
    print(f"Drew and saved a new pool: {len(pool)} records -- {int(pool['is_abnormal'].sum())} abnormal, {int((~pool['is_abnormal']).sum())} normal.")

# Deterministic subset of the pool -- the same 50 records every run, and any of them already
# rendered on Drive (e.g. from a larger-scope run) get reused automatically in Step 3.
normal_in_pool = pool[~pool['is_abnormal']]
abnormal_in_pool = pool[pool['is_abnormal']]
sample = pd.concat([
    normal_in_pool.sample(n=min(len(normal_in_pool), SCOPE_N // 2), random_state=rng_seed),
    abnormal_in_pool.sample(n=min(len(abnormal_in_pool), SCOPE_N - SCOPE_N // 2), random_state=rng_seed),
]).reset_index(drop=True)
print(f"Study sample: {len(sample)} records -- {int(sample['is_abnormal'].sum())} abnormal, {int((~sample['is_abnormal']).sum())} normal.")
sample.head()


### Step 1b -- Patient sex, for the sex-specific QTc threshold

The pool file cached on Drive from v1 doesn't carry `sex` (only `ecg_id`, `filename_lr`, `diagnostic_class`,
`is_abnormal`), so this fetches the small PTB-XL label CSV fresh each session (a few hundred KB, not the
3GB archive) just to look up sex for this sample's 50 records. PTB-XL convention: **0 = male, 1 = female**
(per Wagner et al. 2020's `ptbxl_database.csv` documentation) -- printed below as a sanity check;
double-check against the record if the male/female split looks implausible for the sample.


In [ ]:
BASE = 'https://physionet.org/files/ptb-xl/1.0.3/'
ctx = ssl.create_default_context()
with urllib.request.urlopen(BASE + 'ptbxl_database.csv', context=ctx) as r:
    with open('ptbxl_database_sex.csv', 'wb') as f:
        f.write(r.read())
sex_lookup = pd.read_csv('ptbxl_database_sex.csv')[['ecg_id', 'sex', 'age']]
sample = sample.merge(sex_lookup, on='ecg_id', how='left')
n_male = int((sample['sex'] == 0).sum()); n_female = int((sample['sex'] == 1).sum())
n_missing = int(sample['sex'].isna().sum())
print(f"Sex for this sample: {n_male} male, {n_female} female, {n_missing} missing/unmapped.")
assert n_missing == 0, 'Every sampled record must resolve to a sex for the QTc threshold -- investigate before proceeding.'


## Step 2 -- Fetch the sampled recordings

Streamed directly from PhysioNet via `wfdb` each session -- fast (~50 small files), so not worth
checkpointing separately.


In [ ]:
import wfdb

# LOCAL PATCH: Colab's numpy removed the old np.fromstring binary-read path that
# wfdb's own code still calls internally.
import numpy as np
_original_fromstring = np.fromstring
def _patched_fromstring(string, dtype=float, count=-1, sep=''):
    if sep == '':
        return np.frombuffer(string, dtype=dtype, count=count)
    return _original_fromstring(string, dtype=dtype, count=count, sep=sep)
np.fromstring = _patched_fromstring

os.makedirs('/content/data/full', exist_ok=True)
record_ids = []
sex_by_rid = {}
for i, row in sample.iterrows():
    rel_dir = os.path.dirname(row['filename_lr'])
    rid = os.path.basename(row['filename_lr'])
    if not os.path.exists(f'/content/data/full/{rid}.hea'):
        record = wfdb.rdrecord(rid, pn_dir=f'ptb-xl/1.0.3/{rel_dir}')
        wfdb.wrsamp(rid, fs=record.fs, units=record.units, sig_name=record.sig_name,
                    p_signal=record.p_signal, write_dir='/content/data/full')
    record_ids.append(rid)
    sex_by_rid[rid] = 'female' if row['sex'] == 1 else 'male'
    if (i + 1) % 100 == 0:
        print(f"  fetched {i + 1}/{len(sample)}...")
print(f"{len(record_ids)} recordings ready. Sex map covers {len(sex_by_rid)} records.")


## Step 3 -- Render clean ECG images (skips records already on Drive)

Rendering is the same proven ecg-image-kit setup as the pilot. Clean renders are checkpointed to Drive --
a resumed session only renders records it hasn't rendered before.


In [ ]:
!git clone --depth 1 https://github.com/alphanumericslab/ecg-image-kit.git /content/ecg-image-kit
%cd /content/ecg-image-kit/codes/ecg-image-generator

# LOCAL PATCHES -- identical to the proven pilot notebook. Reused verbatim, not re-derived.
from pathlib import Path

HW_IMPORT = "from HandwrittenText.generate import get_handwritten"
p = Path('gen_ecg_image_from_data.py'); t = p.read_text()
if ("                " + HW_IMPORT) not in t:
    t = t.replace(HW_IMPORT + "\n", "")
    t = t.replace("            if(hw_text):", "            if(hw_text):\n                " + HW_IMPORT)
    p.write_text(t)

p = Path('helper_functions.py'); t = p.read_text()
if "from imgaug import augmenters as iaa\n" in t:
    p.write_text(t.replace("from imgaug import augmenters as iaa\n", ""))

p = Path('CreasesWrinkles/creases.py'); t = p.read_text()
if "import imgaug" in t:
    t = t.replace("import imgaug as ia\n", "").replace("from imgaug import augmenters as iaa\n", "")
    p.write_text(t)

p = Path('ImageAugmentation/augment.py'); t = p.read_text()
if "import imgaug" in t and "def get_augment" in t:
    lines = t.split('\n')
    top_imports = [l for l in lines if 'imgaug' in l and not l.strip().startswith('#')]
    for imp in top_imports:
        if lines.count(imp):
            lines.remove(imp)
    for i, l in enumerate(lines):
        if l.strip().startswith('def get_augment'):
            indent = '    '
            for imp in top_imports:
                lines.insert(i + 1, indent + imp.strip())
            break
    p.write_text('\n'.join(lines))
print("ecg-image-kit patched.")


In [ ]:
requirements_core = '''imageio
imutils
pillow
scikit-image
scikit-learn
scipy
matplotlib
joblib
numpy
pandas
wfdb
pyyaml
qrcode
'''
with open('requirements-core.txt', 'w') as f:
    f.write(requirements_core)
!pip install -q -r requirements-core.txt
print("Image-drawing tool ready.")


In [ ]:
import shutil

import glob

CLEAN_DIR_DRIVE = f'{STUDY_DIR}/images_clean'
os.makedirs('/content/images_clean', exist_ok=True)

already_rendered = {os.path.basename(p) for p in glob.glob(f'{CLEAN_DIR_DRIVE}/*.png')}
needs_render = [rid for rid in record_ids if f'{rid}-0.png' not in already_rendered]

if needs_render:
    print(f"Rendering {len(needs_render)} new record(s) (already have {len(already_rendered)} on Drive)...")
    os.makedirs('/content/data/to_render', exist_ok=True)
    for rid in needs_render:
        for ext in ('.hea', '.dat'):
            shutil.copy(f'/content/data/full/{rid}{ext}', f'/content/data/to_render/{rid}{ext}')
    !python gen_ecg_images_from_data_batch.py -i /content/data/to_render -o /content/images_clean_new --print_header
    for p in glob.glob('/content/images_clean_new/*.png'):
        shutil.copy(p, f'{CLEAN_DIR_DRIVE}/{os.path.basename(p)}')
    shutil.rmtree('/content/data/to_render')
    print(f"Rendered and saved {len(needs_render)} new clean image(s) to Drive.")
else:
    print("All clean images already on Drive -- nothing to render.")

# Sync only this session's scope (record_ids) from Drive to local disk -- not the whole Drive
# cache, which may hold images for a wider pool than the current scope is testing.
n_synced = 0
for rid in record_ids:
    drive_p = f'{CLEAN_DIR_DRIVE}/{rid}-0.png'
    local_p = f'/content/images_clean/{rid}-0.png'
    if os.path.exists(drive_p) and not os.path.exists(local_p):
        shutil.copy(drive_p, local_p)
        n_synced += 1
print(f"{n_synced} image(s) synced from Drive this session; {len(glob.glob('/content/images_clean/*.png'))} clean images available locally in total.")


## Step 4 -- Define the 6 degradation axes (5 levels each) and build the full variant list

Blur, resolution, and JPEG compression are direct pixel/encoding operations. Rotation simulates a tilted
photo. Brightness/shadow simulates uneven lighting (a directional gradient). Grid contrast simulates a faded
or washed-out printout by fading the whole page toward its mean intensity. Each axis has 5 levels; level 0
(clean, no degradation) is shared once per record rather than repeated per axis, matching the
6 x 5 + 1 = 31 variants/record design in `docs/PLAN.md`.


In [ ]:
import cv2

def degrade_blur(img, level):
    k = [3, 7, 11, 15, 21][level - 1]
    return cv2.GaussianBlur(img, (k, k), 0)

def degrade_rotate(img, level):
    angle = [2, 5, 8, 12, 16][level - 1]
    h, w = img.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))

def degrade_jpeg(img, level):
    quality = [50, 30, 15, 8, 3][level - 1]
    ok, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)

def degrade_resolution(img, level):
    factor = [0.75, 0.5, 0.35, 0.25, 0.15][level - 1]
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * factor)), max(1, int(h * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

def degrade_brightness(img, level):
    strength = [0.15, 0.3, 0.45, 0.6, 0.75][level - 1]
    h, w = img.shape[:2]
    gradient = np.tile(np.linspace(1.0, 1.0 - strength, w), (h, 1)).astype(np.float32)
    gradient = cv2.merge([gradient, gradient, gradient])
    return np.clip(img.astype(np.float32) * gradient, 0, 255).astype(np.uint8)

def degrade_grid_contrast(img, level):
    strength = [0.15, 0.3, 0.45, 0.6, 0.75][level - 1]
    mean_val = img.mean()
    return np.clip(img.astype(np.float32) * (1 - strength) + mean_val * strength, 0, 255).astype(np.uint8)

AXES = {
    'blur': degrade_blur,
    'rotate': degrade_rotate,
    'jpeg': degrade_jpeg,
    'resolution': degrade_resolution,
    'brightness': degrade_brightness,
    'gridcontrast': degrade_grid_contrast,
}
LEVELS = [1, 2, 3, 4, 5]

# Only the CURRENT scope's records (the `sample`, not every image that happens to be cached
# locally from Drive -- a wider Drive cache from a previous larger-scope run must not silently
# expand what this run actually tests).
clean_images = [f'/content/images_clean/{rid}-0.png' for rid in record_ids]
missing = [p for p in clean_images if not os.path.exists(p)]
assert not missing, f'{len(missing)} clean image(s) missing for the current sample: {missing[:3]}...'

variant_rows = []
for img_path in clean_images:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    variant_rows.append({'stem': stem, 'axis': 'clean', 'level': 0,
                          'variant_id': f'{stem}__clean0', 'source_path': img_path})
    for axis in AXES:
        for level in LEVELS:
            variant_rows.append({'stem': stem, 'axis': axis, 'level': level,
                                  'variant_id': f'{stem}__{axis}{level}', 'source_path': img_path})

all_variants = pd.DataFrame(variant_rows)
print(f"{len(all_variants)} total variants defined ({len(clean_images)} records x (6 axes x 5 levels + 1 clean) = {len(clean_images) * 31}).")


## Step 5 -- Load the results checkpoint and compute what's left

Separate file from v1 (`results_v2.csv`, not `results.csv`) -- v1's completed, verified 1,550-row
results stay untouched no matter what happens in this notebook.


In [ ]:
RESULTS_PATH = f'{STUDY_DIR}/results_v2.csv'

if os.path.exists(RESULTS_PATH):
    results_so_far = pd.read_csv(RESULTS_PATH)
    done_ids = set(results_so_far['variant_id'])
else:
    results_so_far = pd.DataFrame()
    done_ids = set()

remaining = all_variants[~all_variants['variant_id'].isin(done_ids)].reset_index(drop=True)
n_total, n_done, n_remaining = len(all_variants), len(done_ids), len(remaining)
print(f"{n_done}/{n_total} variants complete ({100*n_done/n_total:.1f}%). {n_remaining} remaining.")

if n_remaining == 0:
    print("\nALL VARIANTS COMPLETE -- nothing left to run. See the summary cells below.")


## Step 6 -- Set up the digitizer (once per session)


In [ ]:
import shutil
if os.path.exists('/content/Open-ECG-Digitizer'):
    shutil.rmtree('/content/Open-ECG-Digitizer')
!git clone https://github.com/Ahus-AIM/Open-ECG-Digitizer.git /content/Open-ECG-Digitizer
%cd /content/Open-ECG-Digitizer
!apt-get -qq install -y git-lfs
!git lfs install
!git lfs pull

path = 'src/utils.py'
with open(path) as f:
    content = f.read()
content = content.replace(
    "from ray.tune import Stopper",
    "try:\n    from ray.tune import Stopper\nexcept ImportError:\n    Stopper = object  # ray not needed for inference"
)
with open(path, 'w') as f:
    f.write(content)
print("Patched: ray import made optional")


In [ ]:
!pip install -q torch torchvision yacs scikit-image tqdm torch-tps
print("Digitizer tools ready.")


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

def write_digitizer_config(images_path, output_path):
    config_content = f'''MODEL:
  class_path: 'src.model.inference_wrapper.InferenceWrapper'
  KWARGS:
    config:
      SIGNAL_EXTRACTOR:
        class_path: 'src.model.signal_extractor.SignalExtractor'
        KWARGS: {{}}
      PERSPECTIVE_DETECTOR:
        class_path: 'src.model.perspective_detector.PerspectiveDetector'
        KWARGS:
          num_thetas: 250
      DEWARPER:
        class_path: 'src.model.dewarper.Dewarper'
        KWARGS:
          abs_peak_threshold: 0.1
      SEGMENTATION_MODEL:
        class_path: 'src.model.unet.UNet'
        weight_path: './weights/unet_weights_07072025.pt'
        KWARGS:
          num_in_channels: 3
          num_out_channels: 4
          dims: [32, 64, 128, 256, 320, 320, 320, 320]
          depth: 2
      CROPPER:
        class_path: 'src.model.cropper.Cropper'
        KWARGS:
          granularity: 80
          percentiles: [0.02, 0.98]
          alpha: 0.85
      PIXEL_SIZE_FINDER:
        class_path: 'src.model.pixel_size_finder.PixelSizeFinder'
        KWARGS:
          min_number_of_grid_lines: 30
          max_number_of_grid_lines: 70
          lower_grid_line_factor: 0.5
      LAYOUT_IDENTIFIER:
        class_path: 'src.model.lead_identifier.LeadIdentifier'
        config_path: 'src/config/lead_layouts_george-moody-2024.yml'
        unet_config_path: 'src/config/lead_name_unet.yml'
        unet_weight_path: './weights/lead_name_unet_weights_07072025.pt'
        KWARGS:
          debug: false
          device: '{device}'
          possibly_flipped: false
          target_num_samples: 10000
          required_valid_samples: 2
    device: '{device}'
    resample_size: 3000
    rotate_on_resample: true
    enable_timing: false
    apply_dewarping: false
DATA:
  images_path: '{images_path}'
  image_extensions: ['.png', '.jpg', '.jpeg', '.JPG']
  save_mode: 'all'
  layout_should_include_substring: false
  output_path: '{output_path}'
  clear_output_dir_if_exists: true
'''
    with open('/content/Open-ECG-Digitizer/batch_config.yml', 'w') as f:
        f.write(config_content)

print("Digitizer config helper ready.")


## Step 7 -- Process remaining variants in checkpointed sub-batches

Each sub-batch: degrade the images, digitize them, compute T1/T2 metrics, then **immediately save the updated
results checkpoint to Drive** -- so a disconnect mid-session loses at most one sub-batch's worth of work
(a few minutes), never the whole session. The loop stops when either all variants are done or the per-session
time budget runs out; re-running the notebook later resumes automatically.


In [ ]:
import time, neurokit2 as nk

SUBBATCH_SIZE = 20  # small so a disconnect loses little (CPU runs ~2 min/image)
SESSION_TIME_BUDGET_S = 10 * 3600  # leaves buffer inside Colab's connection limit
FS_CANONICAL = 1000.0

# Sex-specific QTc prolongation threshold, AHA/ACCF/HRS 2009 (docs/REFERENCES.md ref [52]):
# prolonged QTc is >=460ms in women, >450ms in men. Used only to classify QTc as
# in-range/out-of-range for the T3-style outcome computed in the analysis notebook --
# this notebook only saves the raw values and the classification flag, not an outcome verdict.
QTC_THRESHOLD_MS = {'female': 460.0, 'male': 450.0}

def extract_intervals(signal, fs, hr):
    """PR/QRS/QT/QTc in ms via NeuroKit2 delineation. NaN on any failure --
    unit-tested locally on synthetic ECG (scratchpad/test_intervals.py) before being
    embedded here verbatim."""
    out = {'pr_ms': float('nan'), 'qrs_ms': float('nan'), 'qt_ms': float('nan'), 'qtc_ms': float('nan')}
    try:
        cleaned = nk.ecg_clean(signal, sampling_rate=fs, method='neurokit')
        _, rpeaks = nk.ecg_peaks(cleaned, sampling_rate=fs)
        if rpeaks['ECG_R_Peaks'].sum() < 3:
            return out
        _, waves = nk.ecg_delineate(cleaned, rpeaks, sampling_rate=fs, method='dwt')
        p_on = np.array(waves.get('ECG_P_Onsets', []), dtype=float)
        r_on = np.array(waves.get('ECG_R_Onsets', []), dtype=float)
        r_off = np.array(waves.get('ECG_R_Offsets', []), dtype=float)
        t_off = np.array(waves.get('ECG_T_Offsets', []), dtype=float)

        def mean_diff_ms(a, b):
            n = min(len(a), len(b))
            if n == 0:
                return float('nan')
            d = (b[:n] - a[:n])
            d = d[~np.isnan(d)]
            return float(np.mean(d)) / fs * 1000.0 if len(d) else float('nan')

        out['pr_ms'] = mean_diff_ms(p_on, r_on)
        out['qrs_ms'] = mean_diff_ms(r_on, r_off)
        out['qt_ms'] = mean_diff_ms(r_on, t_off)
        if not np.isnan(out['qt_ms']) and hr and hr > 0 and not np.isnan(hr):
            rr_s = 60.0 / hr
            out['qtc_ms'] = out['qt_ms'] / np.sqrt(rr_s)
    except Exception:
        pass
    return out

def compute_metrics(batch_df, digitized_dir):
    rows = []
    for _, v in batch_df.iterrows():
        gt_rid = v['stem'].split('-')[0]
        record = wfdb.rdrecord(f'/content/data/full/{gt_rid}')
        gt_signal = record.p_signal[:, record.sig_name.index('II')]
        fs_gt = record.fs
        sex = sex_by_rid.get(gt_rid, 'male')  # default only reached if lookup somehow missed -- see Step 1b assert

        csv_path = f'{digitized_dir}/{v["variant_id"]}_timeseries_canonical.csv'
        row = {'variant_id': v['variant_id'], 'record': gt_rid, 'axis': v['axis'], 'level': v['level'], 'sex': sex}

        gt_peaks, _ = nk.ecg_peaks(gt_signal, sampling_rate=fs_gt)
        gt_hr = np.nanmean(nk.ecg_rate(gt_peaks, sampling_rate=fs_gt, desired_length=len(gt_signal)))
        gt_iv = extract_intervals(gt_signal, fs_gt, gt_hr)
        row.update({
            'gt_hr': round(gt_hr, 1),
            'gt_pr_ms': round(gt_iv['pr_ms'], 1) if not np.isnan(gt_iv['pr_ms']) else None,
            'gt_qrs_ms': round(gt_iv['qrs_ms'], 1) if not np.isnan(gt_iv['qrs_ms']) else None,
            'gt_qt_ms': round(gt_iv['qt_ms'], 1) if not np.isnan(gt_iv['qt_ms']) else None,
            'gt_qtc_ms': round(gt_iv['qtc_ms'], 1) if not np.isnan(gt_iv['qtc_ms']) else None,
        })
        row['gt_qtc_abnormal'] = (
            bool(gt_iv['qtc_ms'] > QTC_THRESHOLD_MS[sex]) if not np.isnan(gt_iv['qtc_ms']) else None
        )

        if not os.path.exists(csv_path):
            row.update({'dig_hr': None, 'diff_bpm': None, 'snr_db': None,
                        'dig_pr_ms': None, 'dig_qrs_ms': None, 'dig_qt_ms': None, 'dig_qtc_ms': None,
                        'dig_qtc_abnormal': None, 'diff_pr_ms': None, 'diff_qrs_ms': None,
                        'diff_qt_ms': None, 'diff_qtc_ms': None})
            rows.append(row)
            continue

        digitized = pd.read_csv(csv_path)
        dig_signal = digitized['II'].dropna().to_numpy() / 1000.0

        try:
            dig_peaks, _ = nk.ecg_peaks(dig_signal, sampling_rate=FS_CANONICAL)
            dig_hr = np.nanmean(nk.ecg_rate(dig_peaks, sampling_rate=FS_CANONICAL, desired_length=len(dig_signal)))
        except Exception:
            dig_hr = float('nan')

        dig_iv = extract_intervals(dig_signal, FS_CANONICAL, dig_hr) if len(dig_signal) > 10 else {
            'pr_ms': float('nan'), 'qrs_ms': float('nan'), 'qt_ms': float('nan'), 'qtc_ms': float('nan')}

        diff = abs(gt_hr - dig_hr) if not np.isnan(dig_hr) else None

        def safe_diff(gt_v, dig_v):
            return abs(gt_v - dig_v) if (gt_v is not None and dig_v is not None
                                          and not np.isnan(gt_v) and not np.isnan(dig_v)) else None

        snr_db = None
        if len(dig_signal) > 10:
            n = min(len(gt_signal), len(dig_signal))
            noise = gt_signal[:n] - dig_signal[:n] * (gt_signal[:n].std() / (dig_signal[:n].std() + 1e-9))
            signal_power = np.mean(gt_signal[:n] ** 2)
            noise_power = np.mean(noise ** 2)
            if noise_power > 0:
                snr_db = float(10 * np.log10(signal_power / noise_power))

        row.update({
            'dig_hr': round(dig_hr, 1) if not np.isnan(dig_hr) else None,
            'diff_bpm': round(diff, 2) if diff is not None else None,
            'snr_db': round(snr_db, 1) if snr_db is not None else None,
            'dig_pr_ms': round(dig_iv['pr_ms'], 1) if not np.isnan(dig_iv['pr_ms']) else None,
            'dig_qrs_ms': round(dig_iv['qrs_ms'], 1) if not np.isnan(dig_iv['qrs_ms']) else None,
            'dig_qt_ms': round(dig_iv['qt_ms'], 1) if not np.isnan(dig_iv['qt_ms']) else None,
            'dig_qtc_ms': round(dig_iv['qtc_ms'], 1) if not np.isnan(dig_iv['qtc_ms']) else None,
        })
        row['dig_qtc_abnormal'] = (
            bool(dig_iv['qtc_ms'] > QTC_THRESHOLD_MS[sex]) if not np.isnan(dig_iv['qtc_ms']) else None
        )
        row['diff_pr_ms'] = round(safe_diff(gt_iv['pr_ms'], dig_iv['pr_ms']), 1) if safe_diff(gt_iv['pr_ms'], dig_iv['pr_ms']) is not None else None
        row['diff_qrs_ms'] = round(safe_diff(gt_iv['qrs_ms'], dig_iv['qrs_ms']), 1) if safe_diff(gt_iv['qrs_ms'], dig_iv['qrs_ms']) is not None else None
        row['diff_qt_ms'] = round(safe_diff(gt_iv['qt_ms'], dig_iv['qt_ms']), 1) if safe_diff(gt_iv['qt_ms'], dig_iv['qt_ms']) is not None else None
        row['diff_qtc_ms'] = round(safe_diff(gt_iv['qtc_ms'], dig_iv['qtc_ms']), 1) if safe_diff(gt_iv['qtc_ms'], dig_iv['qtc_ms']) is not None else None
        rows.append(row)
    return pd.DataFrame(rows)

session_start = time.time()
n_processed_this_session = 0

while len(remaining) > 0 and (time.time() - session_start) < SESSION_TIME_BUDGET_S:
    batch = remaining.iloc[:SUBBATCH_SIZE].reset_index(drop=True)

    # Degrade: write this sub-batch's images.
    if os.path.exists('/content/images_degraded'):
        shutil.rmtree('/content/images_degraded')
    os.makedirs('/content/images_degraded', exist_ok=True)
    for _, v in batch.iterrows():
        img = cv2.imread(v['source_path'])
        out_img = img if v['axis'] == 'clean' else AXES[v['axis']](img, v['level'])
        cv2.imwrite(f'/content/images_degraded/{v["variant_id"]}.png', out_img)

    # Digitize this sub-batch.
    if os.path.exists('/content/digitized_batch'):
        shutil.rmtree('/content/digitized_batch')
    write_digitizer_config('/content/images_degraded', '/content/digitized_batch')
    %cd /content/Open-ECG-Digitizer
    t0 = time.time()
    !python -m src.digitize --config /content/Open-ECG-Digitizer/batch_config.yml
    batch_elapsed = time.time() - t0

    # Metrics + checkpoint.
    batch_results = compute_metrics(batch, '/content/digitized_batch')
    results_so_far = pd.concat([results_so_far, batch_results], ignore_index=True)
    results_so_far = results_so_far.drop_duplicates(subset='variant_id', keep='last')
    # Safe save: write a temp file, swap it in atomically, keep a timestamped backup, force a
    # filesystem sync, then READ BACK from Drive and verify -- a lost save must show up now.
    tmp_path = RESULTS_PATH + '.tmp'
    results_so_far.to_csv(tmp_path, index=False)
    os.replace(tmp_path, RESULTS_PATH)
    shutil.copy(RESULTS_PATH, f'{STUDY_DIR}/results_v2_backup_{int(time.time())}.csv')
    os.sync()
    n_on_drive = len(pd.read_csv(RESULTS_PATH))
    assert n_on_drive == len(results_so_far), f'SAVE VERIFY FAILED: Drive has {n_on_drive}, memory has {len(results_so_far)}'
    print(f'  Verified on Drive: {n_on_drive} rows in results_v2.csv')

    remaining = remaining.iloc[SUBBATCH_SIZE:].reset_index(drop=True)
    n_processed_this_session += len(batch)
    rate = batch_elapsed / len(batch)
    elapsed_total = time.time() - session_start
    print(f"Checkpointed {len(batch)} variants ({rate:.1f}s/image). Session total: {n_processed_this_session} done, {len(remaining)} remaining. Session elapsed: {elapsed_total/60:.1f} min.")

n_done_total = len(results_so_far)
n_grand_total = len(all_variants)
print(f"\n=== Session summary ===")
print(f"Processed {n_processed_this_session} variants this session.")
print(f"Overall: {n_done_total}/{n_grand_total} complete ({100*n_done_total/n_grand_total:.1f}%).")
if len(remaining) == 0:
    print("ALL VARIANTS COMPLETE.")
else:
    est_remaining_hours = (rate * len(remaining)) / 3600 if n_processed_this_session > 0 else float('nan')
    print(f"{len(remaining)} variants remaining -- roughly {est_remaining_hours:.1f} more hours at this rate. Re-run this notebook (Runtime -> Run all) to continue.")


## Step 8 -- Results overview (uses everything completed so far, across all sessions)

Safe to run at any point -- it reflects whatever fraction of the sweep is complete, not just this session's
work. Rerun this cell after future sessions to see the picture update.


In [ ]:
results_so_far = pd.read_csv(RESULTS_PATH) if os.path.exists(RESULTS_PATH) else pd.DataFrame()
print(f"{len(results_so_far)}/{len(all_variants)} variants have results so far.")

if len(results_so_far) > 0:
    summary = results_so_far.groupby(['axis', 'level'])['diff_bpm'].agg(['mean', 'count']).round(3)
    print("\nAverage heart-rate error by axis and level (higher level = more degradation):")
    display(summary)
    for col in ('diff_pr_ms', 'diff_qrs_ms', 'diff_qt_ms', 'diff_qtc_ms'):
        n_ok = int(results_so_far[col].notna().sum())
        mean_val = results_so_far[col].mean() if n_ok else float('nan')
        print(f"{col}: {n_ok}/{len(results_so_far)} rows have a value, mean={mean_val:.1f}ms" if n_ok else f"{col}: 0 rows have a value yet")


In [ ]:
import matplotlib.pyplot as plt

if len(results_so_far) > 0:
    axes_present = [a for a in AXES if a in results_so_far['axis'].unique()]
    fig, axs = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
    clean_mean = results_so_far[results_so_far['axis'] == 'clean']['diff_bpm'].mean()
    for ax_name, ax in zip(AXES.keys(), axs.flat):
        sub = results_so_far[results_so_far['axis'] == ax_name]
        if len(sub) == 0:
            continue
        by_level = sub.groupby('level')['diff_bpm'].mean()
        levels_x = [0] + list(by_level.index)
        means_y = [clean_mean] + list(by_level.values)
        ax.plot(levels_x, means_y, marker='o')
        ax.set_title(ax_name)
        ax.set_xlabel('degradation level (0 = clean)')
    axs[0, 0].set_ylabel('mean HR error (bpm)')
    axs[1, 0].set_ylabel('mean HR error (bpm)')
    fig.suptitle('Heart-rate error vs. degradation level, by axis (partial results if sweep incomplete)')
    plt.tight_layout()
    plt.show()
